In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import torch
from hydra.utils import instantiate
from hydra import initialize, compose
import hydra

import wandb

from data.dataManager import DataManager
from model.modelCreator import ModelCreator
from omegaconf import OmegaConf
from scripts.run import setup_model, load_model_instance
from utils.HighLevelFeatsAtlasReg import HighLevelFeatures_ATLAS_regular 
from utils.HLF.atlasgeo import AtlasGeometry, CaloDownsampler, DifferentiableFeatureExtractor, VariableLayerFeatureExtractor, FeatureAdapter, NaiveResampler
from datetime import datetime
import os
from utils.atlas_plots import to_np, make_validation_plots



In [ ]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path="config")
cfg=compose(config_name="config.yaml")
wandb.init(tags = [cfg.data.dataset_name], project=cfg.wandb.project, entity=cfg.wandb.entity, config=OmegaConf.to_container(cfg, resolve=True), mode='disabled')
config = OmegaConf.load(cfg.config_path)
config.gpu_list = cfg.gpu_list
config.load_state = cfg.load_state
self = setup_model(config)
self._model_creator.load_state(config.run_path, self.device)


In [ ]:
regular_binning_path = self._config.data.binning_path
geo_regular = AtlasGeometry(regular_binning_path)

In [ ]:
print(geo_regular.binstart_alpha["2"])
print(geo_regular.binstart_radius["2"])

In [ ]:
self.evaluate_ae(self.data_mgr.val_loader, 0)

In [ ]:
val_data_manager = DataManager(cfg)
self.evaluate_ae(val_data_manager.val_loader, 0)

In [ ]:
regular_binning_path = self._config.data.binning_path
geo_regular = AtlasGeometry(regular_binning_path)

naive_downsampler = NaiveResampler(geo_regular, target_layer_id=2, n_outer_rings=9)
new_geo = naive_downsampler.get_downsampled_geometry()
feature_extractor = VariableLayerFeatureExtractor(new_geo)

In [ ]:
def evaluate_and_plot(data_dict, geo, extractor, output_dir="plots/", device="cpu"):
    """
    Orchestrates the flow: Raw Data -> Fast Extractor -> Adapter -> Existing Plotter
    """
    
    # 1. Setup Geometry & Extractor ONCE
    # (Move to GPU if available)
    
    extractor.eval() # Ensure we are in eval mode
    extractor.to(device)

    populated_adapters = []
    labels = []

    # 2. Process all datasets
    with torch.no_grad(): # No gradients needed for plotting
        for label, (showers, e_inc) in data_dict.items():
            print(f"Extracting features for: {label}...")
            
            # Ensure data is on the correct device
            if not isinstance(showers, torch.Tensor):
                showers = torch.tensor(showers, dtype=torch.float32)
            showers = showers.to(device)

            # --- THE FAST PART ---
            # One forward pass replaces the nested loops
            features = extractor(showers)
            
            # --- THE ADAPTER ---
            # Wrap results to look like the old class
            adapter = FeatureAdapter(features, geo.relevant_layers, e_inc)
            
            populated_adapters.append(adapter)
            labels.append(label)

    # 3. Separate Reference from Models
    # First item is reference (Data/GEANT), rest are models
    adapter_ref = populated_adapters[0]
    list_adapter_models = populated_adapters[1:]
    model_labels = labels[1:]

    # 4. Call existing plotting code
    # It won't know the difference between 'adapter_ref' and the old 'hlf_ref'
    make_validation_plots(adapter_ref, list_adapter_models, model_labels, output_dir=output_dir)

data_to_plot = {
    "GEANT4": (naive_downsampler(self.showers), self.incident_energy),           # The Reference
    "Recon": (naive_downsampler(self.showers_recon), self.incident_energy),     # Model 1

}
wandb_output_path = "/home/leozhu/CaloQuVAE/wandb-outputs"
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

save_dir = os.path.join(
    wandb_output_path, f"plots_{run_timestamp}_binning_simple"
)

evaluate_and_plot(data_to_plot, new_geo, feature_extractor, output_dir=save_dir, device=self.device)

In [ ]:
regular_binning_path = self._config.data.binning_path
optimal_binning_path = "/fast_scratch_1/caloqvae/data/atlas_july31/eta_020/eta_020_default_binning/dataset_combined_positive.hdf5"
# optimal_binning_path = self._config.data.binning_path

# 1. Load Geometries
geo_regular = AtlasGeometry(regular_binning_path)
geo_optimal = AtlasGeometry(optimal_binning_path)

# 2. Init Downsampler (Calculates W once)
downsampler = CaloDownsampler(geo_regular, geo_optimal)

# 3. Init Feature Extractor (Using OPTIMAL geometry)
# We want to extract features from the RESULT of the downsampling
feature_extractor = VariableLayerFeatureExtractor(geo_optimal)

In [ ]:
def verify_geometric_consistency(downsampler, regular_geo, optimal_geo, atol=1e-4):
    """
    Validates the transfer matrix by calculating the expected geometric overlap
    fraction for every single source voxel dynamically.
    """
    print("--- Starting Geometric Consistency Check ---")
    
    # 1. Get the Transfer Matrix Sums (Column Sums)
    # Shape: (N_source_voxels, )
    W = downsampler.transfer_matrix
    if W.is_sparse:
        W = W.to_dense()
    observed_weights = torch.sum(W, dim=0)
    
    # 2. Extract Source Geometry Info (Flattened)
    # We need the r_min, r_max, and layer_id for every source voxel
    src_r_min, src_r_max, _, _, src_layer_ids = downsampler._flatten_geometry(regular_geo)
    
    # 3. Determine R_limit for each layer in Target Geometry
    # We create a lookup table: layer_id -> max_radius
    layer_limits = {}
    for layer_id in optimal_geo.relevant_layers:
        l_str = str(layer_id)
        # The limit is the max radius of the last bin in this layer
        r_end = optimal_geo.binstart_radius[l_str][-1] + optimal_geo.binsize_radius[l_str][-1]
        layer_limits[layer_id] = float(r_end)
        
    # 4. Calculate Expected Weights Vectorized
    expected_weights = torch.zeros_like(observed_weights)
    
    # Loop through unique layers to vectorize the check
    unique_layers = torch.unique(src_layer_ids)
    
    for layer in unique_layers:
        l_idx = int(layer.item())
        if l_idx not in layer_limits:
            continue
            
        R_lim = layer_limits[l_idx]
        
        # Mask for all source voxels in this layer
        mask = (src_layer_ids == l_idx)
        
        # Get their radial bounds
        r_mins = src_r_min[mask]
        r_maxs = src_r_max[mask]
        
        # --- A. Core Voxels (Inside) ---
        # r_max <= R_lim
        core_mask = (r_maxs <= R_lim + atol)
        expected_weights[mask] += core_mask.float() * 1.0
        
        # --- B. Ghost Voxels (Outside) ---
        # r_min >= R_lim
        # expected_weights remains 0.0, so we do nothing
        
        # --- C. Boundary Voxels (Partial) ---
        # r_min < R_lim < r_max
        boundary_mask = (r_mins < R_lim) & (r_maxs > R_lim)
        
        if boundary_mask.any():
            # Calculate overlap fraction: (Limit - Start) / (End - Start)
            overlap_area_proxy = (R_lim**2) - (r_mins[boundary_mask]**2)
            full_area_proxy = (r_maxs[boundary_mask]**2) - (r_mins[boundary_mask]**2)
            fraction = overlap_area_proxy / full_area_proxy            
            # Map back to the full expected_weights tensor
            # We need slightly complex indexing here to put values in right place
            layer_indices = torch.nonzero(mask).squeeze(1)
            boundary_indices = layer_indices[boundary_mask]
            expected_weights[boundary_indices] = fraction

    # 5. The Assertion
    diff = torch.abs(observed_weights - expected_weights)
    max_diff = torch.max(diff).item()
    
    failed = diff > atol
    num_failures = failed.sum().item()
    
    if num_failures == 0:
        print(f"PASSED: All {len(observed_weights)} voxels match geometric expectations.")
        print(f"Max deviation: {max_diff:.6f}")
    else:
        print(f"FAILED: {num_failures} voxels do not match geometric expectations.")
        print(f"Max deviation: {max_diff:.6f}")
        
        # Debug print for first failure
        fail_idx = torch.nonzero(failed)[0].item()
        print(f"\nExample Failure at Index {fail_idx}:")
        print(f"  Layer: {src_layer_ids[fail_idx].item()}")
        print(f"  Radial Range: [{src_r_min[fail_idx]:.1f}, {src_r_max[fail_idx]:.1f}]")
        print(f"  Target Limit: {layer_limits[int(src_layer_ids[fail_idx].item())]:.1f}")
        print(f"  Expected Weight: {expected_weights[fail_idx]:.4f}")
        print(f"  Observed Weight: {observed_weights[fail_idx]:.4f}")

# Run it
verify_geometric_consistency(downsampler, geo_regular, geo_optimal)

In [ ]:
def evaluate_and_plot(data_dict, geo, extractor, output_dir="plots/", device="cpu"):
    """
    Orchestrates the flow: Raw Data -> Fast Extractor -> Adapter -> Existing Plotter
    """
    
    # 1. Setup Geometry & Extractor ONCE
    # (Move to GPU if available)
    
    extractor.eval() # Ensure we are in eval mode
    extractor.to(device)

    populated_adapters = []
    labels = []

    # 2. Process all datasets
    with torch.no_grad(): # No gradients needed for plotting
        for label, (showers, e_inc) in data_dict.items():
            print(f"Extracting features for: {label}...")
            
            # Ensure data is on the correct device
            if not isinstance(showers, torch.Tensor):
                showers = torch.tensor(showers, dtype=torch.float32)
            showers = showers.to(device)

            # --- THE FAST PART ---
            # One forward pass replaces the nested loops
            features = extractor(showers)
            
            # --- THE ADAPTER ---
            # Wrap results to look like the old class
            adapter = FeatureAdapter(features, geo.relevant_layers, e_inc)
            
            populated_adapters.append(adapter)
            labels.append(label)

    # 3. Separate Reference from Models
    # First item is reference (Data/GEANT), rest are models
    adapter_ref = populated_adapters[0]
    list_adapter_models = populated_adapters[1:]
    model_labels = labels[1:]

    # 4. Call existing plotting code
    # It won't know the difference between 'adapter_ref' and the old 'hlf_ref'
    make_validation_plots(adapter_ref, list_adapter_models, model_labels, output_dir=output_dir)

data_to_plot = {
    "GEANT4": (downsampler(self.showers), self.incident_energy),           # The Reference
    "Recon": (downsampler(self.showers_recon), self.incident_energy),     # Model 1

}
wandb_output_path = "/home/leozhu/CaloQuVAE/wandb-outputs"
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

save_dir = os.path.join(
    wandb_output_path, f"plots_{run_timestamp}_binning_test"
)

evaluate_and_plot(data_to_plot, geo_optimal, feature_extractor, output_dir=save_dir, device=self.device)